In [0]:
# =============================================================================
# sql_vs_pipeline_diff  -  READ ONLY. Confirm the SQL->PySpark TRANSLATION is faithful,
# at ZONE level (active / FTA / UTA / FPA / TD) per case.
#
# The 2311/2307/2048 changes are authored as SQL-Server (Bella's v5 active + the 3 archive
# fixes). The live segmentation is PySpark: active in SILVER_ACTIVE_APPEALS (stg_segmentation_states)
# and archive in the 4 ARM pipelines (stg_*_filtered). This diffs, per case, the ZONE the SQL
# assigns vs the ZONE the pipeline assigns.
#
#   PIPELINE side  -> BUILT HERE automatically as CaseNo -> zone from the 5 live tables.
#   SQL side       -> dev exports CaseNo + label (one table/file). Set SQL_RESULT_TBL/PATH.
#                     Labels are mapped to zones via ZONE_MAP (edit if dev's vocabulary differs).
#
# Zone comparison is vocabulary-proof: 'FTA'/'ARIAFTA' both -> FTA, 'Tribunal Decision'/'TD' -> TD,
# and the FTA-query labels 'FT RETAINED - CCD' / 'FT Active Case' -> active (they are EXCLUDED
# from the FTA segment by the fix - that is the whole point of #1290).
#
# Even with NO SQL export it still runs: prints the pipeline zone partition + any case the
# pipeline put in TWO zones (the overlap bug) - a lite reconcile. Single print + saved txt.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth ----
# SQL v5 result (dev export): set ONE. CaseNo + a label/zone column.
SQL_RESULT_TBL  = ""     # e.g. "hive_metastore.test_reporting.seg_sql_v5"
SQL_RESULT_PATH = ""     # e.g. "dbfs:/tmp/seg_sql_v5.csv" (header CaseNo,<label>) or .parquet

# pipeline live tables (CaseNo -> zone is built from these)
ACTIVE_TBL = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"
FTA_TBL    = "hive_metastore.ariadm_arm_fta.stg_appeals_filtered"
UTA_TBL    = "hive_metastore.ariadm_arm_uta.stg_appeals_filtered"
FPA_TBL    = "hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered"
TD_TBL     = "hive_metastore.ariadm_arm_td.stg_td_filtered"   # NOTE: includes dept-519 (huge)
INCLUDE_TD = True        # TD is 1.8M rows; set False for a fast active-vs-appeals-archive check

# raw label -> zone map for the SQL side (lowercased keys). Anything unmapped -> its own value.
ZONE_MAP = {
 "ariafta":"FTA", "fta":"FTA", "ft retained - arm":"FTA", "ft overdue":"FTA",
 "ariauta":"UTA", "uta":"UTA", "ut remitted":"UTA", "ut active":"UTA", "ut retained":"UTA", "ut overdue":"UTA",
 "ariafpa":"FPA", "fpa":"FPA", "file preserved":"FPA", "skeleton case":"FPA",
 "td":"TD", "tribunal decision":"TD",
 "ft retained - ccd":"active", "ft active case":"active",   # EXCLUDED from archive by the fix -> active
}

from pyspark.sql import functions as F
from pyspark.sql.functions import *
REPORT=[]
def log(*a): REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=40):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:100]})")
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")
def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)
def _cases(tbl):
    df=spark.table(tbl); cc=col_ci(df.columns,"CaseNo")
    return df.select(trim(col(cc)).alias("CaseNo")).distinct()
# map a label column to a zone using ZONE_MAP (SQL expression, case-insensitive)
def _to_zone(labelcol):
    e=lower(trim(col(labelcol)))
    m=None
    for k,v in ZONE_MAP.items():
        cond=(e==lit(k))
        m=when(cond,lit(v)) if m is None else m.when(cond,lit(v))
    return m.otherwise(col(labelcol)) if m is not None else col(labelcol)

In [0]:
# ---- CELL 1 : build PIPELINE zone map (CaseNo -> zone) from the 5 live tables ----
log("="*74); log("SQL v5  vs  PIPELINE segmentation - per-case ZONE translation check"); log("="*74)
parts=[
 _cases(ACTIVE_TBL).withColumn("pipe_zone",lit("active")),
 _cases(FTA_TBL).withColumn("pipe_zone",lit("FTA")),
 _cases(UTA_TBL).withColumn("pipe_zone",lit("UTA")),
 _cases(FPA_TBL).withColumn("pipe_zone",lit("FPA")),
]
if INCLUDE_TD: parts.append(_cases(TD_TBL).withColumn("pipe_zone",lit("TD")))
pipe_long=parts[0]
for p in parts[1:]: pipe_long=pipe_long.unionByName(p)
pipe_long=pipe_long.cache()

# pipeline zone partition + internal overlaps (a case the pipeline placed in >1 zone)
log("\n-- PIPELINE zone partition (distinct CaseNos per zone) --")
logdf(pipe_long.groupBy("pipe_zone").agg(countDistinct("CaseNo").alias("cases")).orderBy("pipe_zone"),10)
multi=pipe_long.groupBy("CaseNo").agg(countDistinct("pipe_zone").alias("z"), collect_set("pipe_zone").alias("zones")).filter(col("z")>1).cache()
nmulti=multi.count()
log(f"\n-- cases in MORE THAN ONE pipeline zone (overlap bug): {nmulti} --")
if nmulti:
    logdf(multi.groupBy("zones").count().orderBy(desc("count")),20)
    log("sample:"); logdf(multi.select("CaseNo","zones").limit(30))
# one zone per case (priority active > FTA > UTA > FPA > TD) for the SQL diff
_pri=when(col("pipe_zone")=="active",1).when(col("pipe_zone")=="FTA",2).when(col("pipe_zone")=="UTA",3).when(col("pipe_zone")=="FPA",4).otherwise(5)
from pyspark.sql.window import Window
pipe=(pipe_long.withColumn("_p",_pri)
      .withColumn("_rn",row_number().over(Window.partitionBy("CaseNo").orderBy("_p")))
      .filter(col("_rn")==1).select("CaseNo",col("pipe_zone").alias("pipe_zone")))

In [0]:
# ---- CELL 2 : load SQL side (if provided) and diff at zone level ----
if SQL_RESULT_TBL or SQL_RESULT_PATH:
    if SQL_RESULT_TBL:
        raw=spark.table(SQL_RESULT_TBL); log(f"\nSQL side: table {SQL_RESULT_TBL}")
    else:
        raw=(spark.read.option("header","true").csv(SQL_RESULT_PATH) if SQL_RESULT_PATH.endswith(".csv")
             else spark.read.parquet(SQL_RESULT_PATH)); log(f"\nSQL side: file {SQL_RESULT_PATH}")
    cc=col_ci(raw.columns,"CaseNo")
    lc=col_ci(raw.columns,"zone") or col_ci(raw.columns,"TargetState") or col_ci(raw.columns,"Segment") or col_ci(raw.columns,"label") or col_ci(raw.columns,"State")
    sql=raw.select(trim(col(cc)).alias("CaseNo"), _to_zone(lc).alias("sql_zone")).distinct()
    j=sql.join(pipe,"CaseNo","full_outer")
    match   =j.filter(col("sql_zone").eqNullSafe(col("pipe_zone")) & col("sql_zone").isNotNull())
    mismatch=j.filter(col("sql_zone").isNotNull() & col("pipe_zone").isNotNull() & ~col("sql_zone").eqNullSafe(col("pipe_zone")))
    sqlonly =j.filter(col("pipe_zone").isNull())
    pipeonly=j.filter(col("sql_zone").isNull())
    nM,nX,nS,nP=match.count(),mismatch.count(),sqlonly.count(),pipeonly.count()
    log("\n"+"-"*74)
    log(f"MATCH (same zone both sides):    {nM}")
    log(f"MISMATCH (different zone):        {nX}   <-- translation bug")
    log(f"SQL-only (in SQL, not pipeline): {nS}   <-- pipeline missing these")
    log(f"PIPELINE-only (not in SQL):      {nP}   <-- pipeline has extra")
    if nX:
        log("\n>>> MISMATCH cross-tab (sql_zone -> pipe_zone):")
        logdf(mismatch.groupBy("sql_zone","pipe_zone").count().orderBy(desc("count")),40)
        log("\nsample mismatches:"); logdf(mismatch.select("CaseNo","sql_zone","pipe_zone").limit(40))
    if nS: log("\nsample SQL-only:");      logdf(sqlonly.select("CaseNo","sql_zone").limit(20))
    if nP: log("\nsample PIPELINE-only:"); logdf(pipeonly.select("CaseNo","pipe_zone").limit(20))
    SQL_DONE=True; _tot=nX+nS+nP
else:
    log("\n(no SQL export set - showing pipeline partition only. Set SQL_RESULT_TBL/PATH to diff vs SQL v5.)")
    SQL_DONE=False; _tot=nmulti

In [0]:
# ---- CELL 3 : verdict + single print ----
log("\n"+"="*74)
if SQL_DONE:
    ok=(_tot==0)
    log(f">>> VERDICT: {'PASS - pipeline zone assignment matches the SQL v5 for every case' if ok else 'FAIL - '+str(_tot)+' case(s) differ (see cross-tab / only-lists)'}")
    log("READ: MISMATCH cross-tab pinpoints WHICH zone pair is wrong -> maps to the .when() block dev mis-translated.")
else:
    ok=(nmulti==0)
    log(f">>> PIPELINE-ONLY VERDICT: {'clean partition - every case in exactly one zone' if ok else str(nmulti)+' case(s) in >1 zone (overlap bug - see above)'}")
    log("READ: provide the SQL v5 export (SQL_RESULT_TBL/PATH) to also verify the SQL->PySpark translation per case.")
log("      NOTE: SQL uses the run's retention ref date (staging cut '2026-06-05'); the pipeline that built")
log("      these tables must use the SAME reference, else CCD/archive boundaries differ.")
log("      NOTE: 'FT RETAINED - CCD' / 'FT Active Case' map to ZONE 'active' (excluded from FTA by the 2307 fix).")
full="\n".join(REPORT)
try:
    from datetime import datetime
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/sql_vs_pipeline/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/sql_vs_pipeline_diff.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save failed: {str(e)[:80]})"
print(full)